# Distance from data centers to the recorded electricity grid

This notebook calculates the shortest geometric distance from each data center
to the nearest electricity-grid line in `grid/grid.gpkg`.

The grid layer combines lines labelled as `openstreetmap` and `gridfinder`.
Because the file does not report voltage, the results describe proximity to
**any recorded grid line**, rather than specifically to high-voltage
transmission infrastructure.

The analysis retains data centers commissioned by 2024 with valid coordinates,
matching the spatial sample used by the main analysis. Distances are calculated
from points to line geometries in local azimuthal-equidistant projections.

In [ ]:
import os
import warnings

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

# Keep SVG text editable in Adobe Illustrator.
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["svg.hashsalt"] = "data-center-grid-distance"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ============================================================
# Project paths
# ============================================================

current = os.getcwd()
while os.path.basename(current) != "Data_center_and_fossil_energy_Replication":
    parent = os.path.dirname(current)
    if parent == current:
        raise RuntimeError(
            "Could not find the Data_center_and_fossil_energy_Replication root."
        )
    current = parent

BASE_PATH = current
RAW = os.path.join(BASE_PATH, "Data", "raw")
TEMP = os.path.join(BASE_PATH, "Data", "temp")
USE = os.path.join(BASE_PATH, "Data", "use")
FIGURES = os.path.join(BASE_PATH, "Results", "Figures")
TABLES = os.path.join(BASE_PATH, "Results", "Tables")

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    os.makedirs(path, exist_ok=True)

print(f"BASE_PATH: {BASE_PATH}")

# ============================================================
# Inputs and temporary outputs
# ============================================================

DATA_CENTER_FILE = os.path.join(RAW, "SPGlobal_Export.xlsx")
GRID_FILE = os.path.join(RAW, "grid", "grid.gpkg")

# Store computational caches in the standard temporary-data directory.
CACHE_FILE = os.path.join(
    TEMP,
    "data_center_nearest_grid_distance.csv",
)
GRID_LAYER = "grid"

# Small tiles keep each spatial query manageable. Search radii are expanded
# only for data centers without a grid line in the preceding radius.
TILE_SIZE_DEG = 5
SEARCH_RADII_KM = [100, 300, 1000]
RECALCULATE = False

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})


## Inspect the grid layer

The GeoPackage is queried through its spatial index; the full 3.7-million-line
layer is not loaded into memory.

In [2]:
grid_info = pyogrio.read_info(GRID_FILE, layer=GRID_LAYER)

grid_summary = pd.DataFrame({
    "Property": [
        "Layer",
        "Geometry type",
        "Coordinate system",
        "Number of line features",
        "Attributes",
    ],
    "Value": [
        grid_info["layer_name"],
        grid_info["geometry_type"],
        str(grid_info["crs"]),
        f'{grid_info["features"]:,}',
        ", ".join(grid_info["fields"]),
    ],
})

display(grid_summary)


,Property,Value
0,Layer,grid
1,Geometry type,LineString
2,Coordinate system,EPSG:4326
3,Number of line features,"3,678,243"
4,Attributes,source


## Load the data-center sample

In [3]:
dc = pd.read_excel(
    DATA_CENTER_FILE,
    sheet_name="Sheet1",
)

for column in ["LATITUDE", "LONGITUDE", "YR_BUILT"]:
    dc[column] = pd.to_numeric(dc[column], errors="coerce")

dc = dc[
    dc["LATITUDE"].between(-90, 90)
    & dc["LONGITUDE"].between(-180, 180)
    & dc["YR_BUILT"].notna()
    & dc["YR_BUILT"].le(2024)
].copy()

dc["YR_BUILT"] = dc["YR_BUILT"].astype(int)
dc = dc.drop_duplicates("PPTY_KEY").reset_index(drop=True)

dc["Commissioning period"] = pd.cut(
    dc["YR_BUILT"],
    bins=[-np.inf, 2005, 2015, 2019, 2024],
    labels=["Before 2006", "2006-2015", "2016-2019", "2020-2024"],
)

print(f"Data centers retained: {len(dc):,}")
display(
    dc["Commissioning period"]
    .value_counts(sort=False)
    .rename("N data centers")
    .to_frame()
)


Data centers retained: 9,031


,N data centers
Commissioning period,
Before 2006,1217
2006-2015,3379
2016-2019,1693
2020-2024,2742


## Calculate nearest point-to-line distances

For each 5-degree tile, nearby grid lines are read using the GeoPackage spatial
index. Distances are measured in a local azimuthal-equidistant projection.
Searches expand from 100 km to 1,000 km only when necessary.

Set `RECALCULATE = True` to overwrite the cached results.

In [ ]:
def local_aeqd_crs(longitude, latitude):
    return (
        f"+proj=aeqd +lat_0={latitude:.8f} +lon_0={longitude:.8f} "
        "+datum=WGS84 +units=m +no_defs"
    )


def padded_bbox(points, radius_km):
    minx, miny, maxx, maxy = points.total_bounds
    lat_pad = radius_km / 110.6

    max_abs_lat = min(
        89.0,
        max(abs(miny - lat_pad), abs(maxy + lat_pad)),
    )
    lon_scale = max(np.cos(np.radians(max_abs_lat)), 0.02)
    lon_pad = min(180.0, radius_km / (111.32 * lon_scale))

    return (
        max(-180.0, minx - lon_pad),
        max(-90.0, miny - lat_pad),
        min(180.0, maxx + lon_pad),
        min(90.0, maxy + lat_pad),
    )


def nearest_grid_for_tile(points, radius_km):
    bbox = padded_bbox(points, radius_km)

    lines = pyogrio.read_dataframe(
        GRID_FILE,
        layer=GRID_LAYER,
        bbox=bbox,
        columns=["source"],
    )

    if lines.empty:
        return pd.DataFrame(
            columns=["_row_id", "distance_km", "nearest_grid_source"]
        )

    center_lon = float(points.geometry.x.mean())
    center_lat = float(points.geometry.y.mean())
    local_crs = local_aeqd_crs(center_lon, center_lat)

    points_local = points.to_crs(local_crs)
    lines_local = lines.to_crs(local_crs)

    joined = gpd.sjoin_nearest(
        points_local[["_row_id", "geometry"]],
        lines_local[["source", "geometry"]],
        how="left",
        max_distance=radius_km * 1000,
        distance_col="distance_m",
    )

    joined = (
        joined.dropna(subset=["distance_m"])
        .sort_values("distance_m")
        .drop_duplicates("_row_id")
    )

    return pd.DataFrame({
        "_row_id": joined["_row_id"].astype(int),
        "distance_km": joined["distance_m"].to_numpy() / 1000,
        "nearest_grid_source": joined["source"].to_numpy(),
    })


def calculate_nearest_grid_distances(data_centers):
    points = gpd.GeoDataFrame(
        data_centers.copy(),
        geometry=gpd.points_from_xy(
            data_centers["LONGITUDE"],
            data_centers["LATITUDE"],
        ),
        crs="EPSG:4326",
    )

    points["_row_id"] = np.arange(len(points))
    points["_tile_x"] = np.floor(
        (points["LONGITUDE"] + 180) / TILE_SIZE_DEG
    ).astype(int)
    points["_tile_y"] = np.floor(
        (points["LATITUDE"] + 90) / TILE_SIZE_DEG
    ).astype(int)

    output = points[["_row_id"]].copy().set_index("_row_id")
    output["distance_km"] = np.nan
    output["nearest_grid_source"] = pd.NA

    tile_groups = list(points.groupby(["_tile_x", "_tile_y"], sort=False))
    print(f"Spatial tiles to process: {len(tile_groups):,}")

    for tile_number, (_, tile_points) in enumerate(tile_groups, start=1):
        unresolved = tile_points.copy()

        for radius_km in SEARCH_RADII_KM:
            if unresolved.empty:
                break

            matches = nearest_grid_for_tile(unresolved, radius_km)
            if matches.empty:
                continue

            matches = matches.set_index("_row_id")
            match_ids = matches.index.to_numpy()
            output.loc[match_ids, "distance_km"] = matches["distance_km"]
            output.loc[match_ids, "nearest_grid_source"] = (
                matches["nearest_grid_source"]
            )

            unresolved = unresolved[
                ~unresolved["_row_id"].isin(match_ids)
            ]

        if tile_number % 25 == 0 or tile_number == len(tile_groups):
            matched_n = output["distance_km"].notna().sum()
            print(
                f"Processed {tile_number:,}/{len(tile_groups):,} tiles; "
                f"matched {matched_n:,}/{len(points):,} data centers"
            )

    result = points.drop(
        columns=["geometry", "_tile_x", "_tile_y"]
    ).merge(
        output.reset_index(),
        on="_row_id",
        how="left",
    )

    result["distance_status"] = np.where(
        result["distance_km"].notna(),
        "Matched",
        f"No recorded line within {max(SEARCH_RADII_KM):,} km",
    )

    return result.drop(columns="_row_id")


if os.path.exists(CACHE_FILE) and not RECALCULATE:
    distance_df = pd.read_csv(CACHE_FILE)
    print(f"Loaded cached results: {CACHE_FILE}")
else:
    distance_df = calculate_nearest_grid_distances(dc)
    distance_df.to_csv(CACHE_FILE, index=False)
    print(f"Saved distance results: {CACHE_FILE}")

print("\nDistance matching status:")
display(distance_df["distance_status"].value_counts(dropna=False).to_frame("N"))


## Descriptive distribution

In [ ]:
matched = distance_df.dropna(subset=["distance_km"]).copy()

quantile_levels = [0, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1]
quantile_table = (
    matched["distance_km"]
    .quantile(quantile_levels)
    .rename_axis("Quantile")
    .reset_index(name="Distance to nearest grid line (km)")
)

quantile_table["Quantile"] = quantile_table["Quantile"].map(
    lambda value: f"{value:.0%}"
)

thresholds_km = [0.5, 1, 2, 5, 10, 25, 50, 100]
threshold_table = pd.DataFrame({
    "Distance threshold (km)": thresholds_km,
    "N data centers": [
        int((matched["distance_km"] <= threshold).sum())
        for threshold in thresholds_km
    ],
    "Share of matched data centers (%)": [
        100 * (matched["distance_km"] <= threshold).mean()
        for threshold in thresholds_km
    ],
})

print("Distance quantiles")
display(quantile_table.round(3))

print("Cumulative shares within selected thresholds")
display(threshold_table.round(2))

print("Nearest recorded grid source")
display(
    matched["nearest_grid_source"]
    .value_counts(dropna=False)
    .rename("N data centers")
    .to_frame()
)


In [ ]:
period_summary = (
    matched.groupby("Commissioning period", observed=False)
    .agg(
        N=("PPTY_KEY", "nunique"),
        Mean_km=("distance_km", "mean"),
        Median_km=("distance_km", "median"),
        P90_km=("distance_km", lambda x: x.quantile(0.90)),
        Within_5km_pct=("distance_km", lambda x: 100 * (x <= 5).mean()),
        Within_25km_pct=("distance_km", lambda x: 100 * (x <= 25).mean()),
    )
    .reset_index()
)

display(period_summary.round(2))


## Distribution figure

In [ ]:
from matplotlib.ticker import PercentFormatter

# ============================================================
# Nature-style, Illustrator-friendly distance distribution
# ============================================================

GRID_COLOR = "#B8B8B8"
DC_COLOR = "#DE1A58"
TEXT_COLOR = "#222222"

distance = matched["distance_km"].dropna().to_numpy()
display_limit = np.quantile(distance, 0.99)

plot_values = distance[distance <= display_limit]
sorted_distance = np.sort(plot_values)
cumulative_share = np.searchsorted(
    np.sort(distance),
    sorted_distance,
    side="right",
) / len(distance)

median_distance = np.median(distance)
share_1km = np.mean(distance <= 1)
share_2km = np.mean(distance <= 2)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
})

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7.2, 2.75),
    gridspec_kw={"wspace": 0.34},
)

# ------------------------------------------------------------
# a. Histogram
# ------------------------------------------------------------

_, _, histogram_patches = axes[0].hist(
    plot_values,
    bins=45,
    color=GRID_COLOR,
    edgecolor="white",
    linewidth=0.3,
)

# Disable clipping because every displayed bar is already inside the axes.
for patch in histogram_patches:
    patch.set_clip_on(False)

median_line = axes[0].axvline(
    median_distance,
    color=DC_COLOR,
    linewidth=1.2,
    linestyle="--",
)
median_line.set_clip_on(False)

axes[0].text(
    median_distance + 0.03 * display_limit,
    axes[0].get_ylim()[1] * 0.93,
    f"Median = {median_distance:.2f} km",
    color=DC_COLOR,
    fontsize=8,
    ha="left",
    va="top",
)

axes[0].set_xlim(0, display_limit)
axes[0].set_xlabel("Distance to nearest recorded grid line (km)")
axes[0].set_ylabel("Number of data centers")

# ------------------------------------------------------------
# b. Cumulative distribution
# ------------------------------------------------------------

# Only the visible 0-99th-percentile section is written to the SVG. This
# prevents a hidden line extending to distant outliers from appearing when
# the curve is moved in Illustrator.
cdf_line, = axes[1].plot(
    sorted_distance,
    cumulative_share,
    color=DC_COLOR,
    linewidth=1.5,
    clip_on=False,
)

for threshold, share in [(1, share_1km), (2, share_2km)]:
    if threshold <= display_limit:
        point, = axes[1].plot(
            threshold,
            share,
            marker="o",
            markersize=4,
            markerfacecolor="white",
            markeredgecolor=DC_COLOR,
            markeredgewidth=1,
            linestyle="none",
            clip_on=False,
            zorder=3,
        )

        vertical = axes[1].plot(
            [threshold, threshold],
            [0, share],
            color=GRID_COLOR,
            linewidth=0.6,
            linestyle=":",
            clip_on=False,
        )[0]

        horizontal = axes[1].plot(
            [0, threshold],
            [share, share],
            color=GRID_COLOR,
            linewidth=0.6,
            linestyle=":",
            clip_on=False,
        )[0]

        axes[1].annotate(
            f"{share:.1%} within {threshold} km",
            xy=(threshold, share),
            xytext=(-5, -13 if threshold == 1 else 8),
            textcoords="offset points",
            ha="right",
            va="center",
            fontsize=7.5,
            color=TEXT_COLOR,
        )

axes[1].set_xlim(0, display_limit)
axes[1].set_ylim(0, 1.015)
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1].set_xlabel("Distance to nearest recorded grid line (km)")
axes[1].set_ylabel("Cumulative share of data centers")

# ------------------------------------------------------------
# Shared styling
# ------------------------------------------------------------

for label, ax in zip(["a", "b"], axes):
    ax.text(
        -0.14,
        1.03,
        label,
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        ha="left",
        va="bottom",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(TEXT_COLOR)
    ax.spines["bottom"].set_color(TEXT_COLOR)
    ax.tick_params(direction="out", colors=TEXT_COLOR)
    ax.grid(False)

fig.subplots_adjust(
    left=0.10,
    right=0.99,
    bottom=0.20,
    top=0.95,
)

figure_path = os.path.join(
    FIGURES,
    "data_center_distance_to_grid_distribution.svg",
)

fig.savefig(
    figure_path,
    format="svg",
    pad_inches=0.02,
)

plt.show()
print(f"Saved: {figure_path}")


## Global overlap of data centers and recorded grid lines

For visualization only, the 3.7-million-feature grid layer is rasterized to a
0.1-degree global grid. The distance calculations above continue to use the
original vector line geometries.

In [ ]:
import subprocess

import matplotlib.pyplot as plt
import rasterio

from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

GRID_MAP_RASTER = os.path.join(
    TEMP,
    "grid_lines_global_0p1deg.tif",
)

# ============================================================
# Rasterize grid lines for visualization only
# ============================================================

if not os.path.exists(GRID_MAP_RASTER):
    command = [
        "gdal_rasterize",
        "-burn", "1",
        "-ot", "Byte",
        "-init", "0",
        "-a_nodata", "0",
        "-at",
        "-te", "-175.5", "-53.5", "178.5", "72",
        "-tr", "0.1", "0.1",
        "-l", GRID_LAYER,
        str(GRID_FILE),
        str(GRID_MAP_RASTER),
    ]

    subprocess.run(command, check=True)
    print(f"Created display raster: {GRID_MAP_RASTER}")

with rasterio.open(GRID_MAP_RASTER) as src:
    grid_display = src.read(1, masked=True)
    grid_extent = [
        src.bounds.left,
        src.bounds.right,
        src.bounds.bottom,
        src.bounds.top,
    ]

# ============================================================
# Nature-style global map
# ============================================================

GRID_COLOR = "#B8B8B8"
DC_COLOR = "#DE1A58"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.6,
})

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.set_facecolor("#FAFAFA")

# Recorded electricity-grid lines
ax.imshow(
    grid_display,
    extent=grid_extent,
    origin="upper",
    cmap=ListedColormap([GRID_COLOR]),
    interpolation="nearest",
    alpha=0.62,
    zorder=1,
    rasterized=True,
)

# All data centers, without period distinction
ax.scatter(
    dc["LONGITUDE"],
    dc["LATITUDE"],
    s=7,
    color=DC_COLOR,
    alpha=0.72,
    edgecolors="white",
    linewidths=0.18,
    zorder=2,
    rasterized=True,
)

ax.set_xlim(grid_extent[0], grid_extent[1])
ax.set_ylim(grid_extent[2], grid_extent[3])
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")

for spine in ax.spines.values():
    spine.set_visible(False)

legend_handles = [
    Line2D(
        [0], [0],
        marker="s",
        linestyle="none",
        markersize=6,
        markerfacecolor=GRID_COLOR,
        markeredgecolor="none",
        label="Recorded grid lines",
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="none",
        markersize=5,
        markerfacecolor=DC_COLOR,
        markeredgecolor="white",
        markeredgewidth=0.3,
        label="Data centers",
    ),
]

ax.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=2,
    frameon=False,
    fontsize=8,
    handletextpad=0.5,
    columnspacing=1.6,
)

fig.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.99,
    bottom=0.12,
)

map_path = os.path.join(
    FIGURES,
    "global_data_centers_and_recorded_grid_lines.svg",
)

fig.savefig(
    map_path,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.02,
    dpi=300,
)

plt.show()
print(f"Saved: {map_path}")


## Interpretation

Use the median, upper quantiles and cumulative shares to assess whether data
centers are generally close to recorded grid infrastructure. This is a
descriptive validation of grid accessibility. It does not identify voltage
level, available interconnection capacity, contractual supply relationships or
the particular generating unit serving a data center.